# Дипломный проект
## Тема: Определение уязвимых групп населения и факторов бедности в регионах России

**Выполнил:** Трегубов Сергей Юрьевич
**Дата:** Март 2026
**Набор данных:** Архивные данные Росстата, Генпрокуратуры и других ведомств за 1990-2022 гг.

## Оглавление
1. [Постановка задачи](#section1)
2. [Исследовательский анализ данных (EDA)](#section2)
3. [Формирование единого датасета](#section3)
4. [Анализ факторов бедности](#section4)
5. [Кластеризация регионов](#section5)
6. [Описание уязвимых групп](#section6)
7. [Выводы и рекомендации](#section7)
---

<div id="section1"></div>

### 1. Постановка задачи

#### Контекст исследования

Данный дипломный проект выполнен на основе брифа, предоставленного Лабораторией исследований гражданского общества. Заказчиком выступает молодая некоммерческая организация (НКО), планирующая открывать свои филиалы в регионах России для помощи малообеспеченным слоям населения.

Согласно опросу "инФОМ" от декабря 2021 года, около 36% россиян испытывают финансовые трудности: 27% хватает денег только на еду, а ещё 9% не могут позволить себе полноценное питание. При этом официальная статистика (Росстат) оценивает необходимую долю расходов на продукты питания в 36% от среднемесячных расходов, на ЖКХ - около 10%, на лекарства - 4%.

#### Цель работы

Провести анализ социально-экономического положения регионов России для выявления территорий и групп населения, наиболее остро нуждающихся в поддержке, а также определить ключевые факторы, связанные с уровнем бедности.

#### Задачи исследования

1. **Кластеризовать регионы России** по уровню социального неблагополучия и определить территории, наиболее нуждающиеся в открытии филиалов НКО.

2. **Описать группы населения, сталкивающиеся с бедностью**:
   - оценить долю детей, пенсионеров, инвалидов среди малоимущих;
   - выявить региональные особенности состава уязвимых групп.

3. **Определить факторы, влияющие на уровень бедности**:
   - проверить связь с демографической структурой (доля детей, пенсионеров);
   - оценить влияние экономических показателей (ВРП, производство, розничная торговля);
   - проанализировать связь с социальными проблемами (преступность, заболеваемость, жилищные условия).

#### Источники данных

В работе используются открытые данные Росстата, Генпрокуратуры, Минздрава и других ведомств за период 1990-2022 годов:
- показатели бедности и доходов населения;
- демографические характеристики (численность, рождаемость, инвалидность);
- экономические показатели (ВРП, производство, розничная торговля);
- социальные индикаторы (преступность, заболеваемость, жилищные условия);
- данные о расходах бюджетов на социальную политику.

#### Ограничения исследования

- Исследование выполнено в 2026 году на архивных данных, поэтому выводы отражают ситуацию на момент сбора данных (преимущественно 2020 год) и могут требовать корректировки с учётом текущей экономической и демографической динамики.
- В связи с изменением методик расчёта некоторых показателей и наличием пропусков в данных, результаты следует интерпретировать как выявление устойчивых тенденций и взаимосвязей, а не как точные прогнозные оценки.

<div id="section2"></div>

### 2. Исследовательский анализ данных (EDA)

**Цель раздела:** Понять структуру данных, выявить пропуски, оценить распределения ключевых показателей.

#### 2.1 Подготовка окружения
- Импорт библиотек
- Настройки отображения
- Определение пути к данным

In [1]:
# 2.1 Подготовка окружения

# Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import re
warnings.filterwarnings('ignore')

# Настройки отображения
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.2f}'.format)

# Настройки графиков
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Путь к данным
DATA_PATH = "data/"

#### 2.2 Обзор доступных источников
- Список всех файлов и их структура
- Временные периоды, за которые есть данные
- Проблемы с наименованиями регионов

В нашем распоряжении находятся следующие файлы с данными:

#### Основные социально-экономические показатели

| Файл | Описание | Период | Источник |
|------|----------|--------|----------|
| `population.xlsx` | Численность населения по регионам и федеральным округам | 1999-2022 | Росстат |
| `poverty_percent_by_regions_1992_2020.csv` | Доля населения с доходами ниже прожиточного минимума | 1992-2020 | Росстат |
| `cash_real_income_wages_2015_2020.xlsx` | Среднедушевые доходы, реальные доходы, зарплаты | 2015-2020 | Росстат |
| `gross_regional_product_1996_2020.xls` | ВРП на душу населения | 1996-2020 | Росстат |
| `retail_turnover_per_capita_2000_2021.xls` | Оборот розничной торговли на душу населения | 2000-2021 | Росстат |

#### Демография и социальная защита

| Файл | Описание | Период | Источник |
|------|----------|--------|----------|
| `child_mortality_rural_1990_2021.xls` | Младенческая смертность в сельской местности | 1990-2021 | Росстат |
| `child_mortality_urban_1990_2021.xls` | Младенческая смертность в городской местности | 1990-2021 | Росстат |
| `disabled_total_by_age_2017_2022.csv` | Численность инвалидов по возрастным группам | 2017-2022 | Росстат |
| `newborn_2006_2022_monthly.csv` | Число родившихся помесячно | 2006-2022 | Росстат |
| `workers.csv` | Отношение занятых к численности трудоспособного населения | 2012-2020 | Росстат |

#### Здравоохранение

| Файл | Описание | Период | Источник |
|------|----------|--------|----------|
| `morbidity_2005_2020_age_disease.xls` | Заболеваемость по возрастным группам и группам заболеваний | 2005-2020 | Минздрав |
| `drug_alco.xlsx` | Заболеваемость алкоголизмом и наркоманией (впервые выявленные) | 2005-2018 | Минздрав |

#### Жилищные условия и бюджет

| Файл | Описание | Период | Источник |
|------|----------|--------|----------|
| `housing_2020.xlsx` | Характеристика жилищных условий домохозяйств | 2020 | Росстат |
| `welfare_expense_share_2015_2020.xlsx` | Доля расходов на социальную политику в бюджете региона | 2015-2020 | Росстат |

#### Данные о преступности (папка crimes)

В папке `crimes` содержится 78 файлов с помесячной статистикой преступлений за 2016-2022 гг.:

| Год | Количество файлов | Формат |
|-----|-------------------|--------|
| 2016 | 12 | `4-EGS_Razdel_4_*.xls` |
| 2017 | 12 | `4-EGS_Razdel_4_*.xls` |
| 2018 | 12 | `4-EGS_Razdel_4_*.xls` |
| 2019 | 12 | `4-EGS_Razdel_4_*.xls` |
| 2020 | 12 | `4-EGS_Razdel_4_*.xls` |
| 2021 | 12 | `4-EGS_Razdel_5_*.xls` |
| 2022 | 6 | `4-EGS_Razdel_5_*.xls` |

Данные содержат информацию о преступлениях, совершённых:
- несовершеннолетними
- ранее совершавшими преступления (в т.ч. ранее судимыми)
- группой лиц (разные формы групповых преступлений)
- в состоянии опьянения
- членами семьи (с 2021 года)
- иностранными гражданами (с 2021 года)

**Важно:** файлы содержат накопленные данные за год (январь, январь-февраль, ...), формат представления изменился в 2021 году.

#### Проблемы, требующие внимания

При предварительном анализе данных выявлены следующие особенности:

1. **Разные форматы дат** - в разных файлах годы могут быть представлены как числа, строки или индексы колонок.

2. **Неунифицированные названия регионов** - один и тот же регион может называться по-разному в разных файлах (например, "г. Москва", "Москва", "Город Москва").

3. **Сложная структура файлов** - некоторые файлы содержат множественные листы, технические строки, объединённые ячейки.

4. **Особенности Тюменской и Архангельской областей** - эти регионы включают автономные округа, которые могут учитываться как отдельно, так и в составе области.

5. **Накопительный характер данных о преступлениях** - требует преобразования помесячных данных в годовые.

6. **Изменение методик подсчёта** - некоторые показатели могли менять методику расчёта за период наблюдения.

В следующих разделах мы последовательно разберём каждый файл, приведём данные к единому формату и создадим чистый датасет для анализа.

In [2]:
# Получаем список всех файлов в директории data
all_files = [f for f in os.listdir(DATA_PATH) if os.path.isfile(DATA_PATH + f)]
print(f"\nВсего файлов в data: {len(all_files)}")
print("\nСписок файлов:")
for f in sorted(all_files):
    size = os.path.getsize(DATA_PATH + f) / 1024
    print(f"  {f} ({size:.1f} КБ)")

# Папка crimes
crimes_path = DATA_PATH + 'crimes/'
if os.path.exists(crimes_path):
    crime_files = [f for f in os.listdir(crimes_path) if os.path.isfile(crimes_path + f)]
    print(f"\nПапка crimes: {len(crime_files)} файлов")
    print("Примеры файлов (первые 5):")
    for f in sorted(crime_files)[:5]:
        size = os.path.getsize(crimes_path + f) / 1024
        print(f"  {f} ({size:.1f} КБ)")

print("\n" + "="*50)
print("АНАЛИЗ СТРУКТУРЫ ФАЙЛОВ")
print("="*50)

# Функция для быстрого просмотра файла
def inspect_file(filepath, filename, n_rows=5):
    """Показывает структуру файла"""
    print(f"\n{'-'*40}")
    print(f"Файл: {filename}")
    print(f"{'-'*40}")
    
    try:
        if filename.endswith('.csv'):
            # Пробуем разные кодировки и разделители
            try:
                df = pd.read_csv(filepath, nrows=n_rows, encoding='utf-8')
            except:
                df = pd.read_csv(filepath, nrows=n_rows, encoding='cp1251')
            
            print(f"Тип: CSV")
            print(f"Размерность: {df.shape}")
            print(f"Колонки: {df.columns.tolist()}")
            print("\nПервые строки:")
            display(df)
            
        elif filename.endswith(('.xls', '.xlsx')):
            # Для Excel показываем все листы
            xl = pd.ExcelFile(filepath)
            print(f"Тип: Excel")
            print(f"Листы: {xl.sheet_names}")
            
            # Показываем первый лист
            df = pd.read_excel(filepath, sheet_name=0, nrows=n_rows, header=None)
            print(f"\nПервый лист (первые {n_rows} строк, без заголовков):")
            display(df)
            
            # Если есть другие листы, показываем их названия
            if len(xl.sheet_names) > 1:
                print(f"\nОстальные листы: {xl.sheet_names[1:]}")
                
    except Exception as e:
        print(f"ОШИБКА при чтении: {e}")

# Анализируем файлы в data
print("\n--- ФАЙЛЫ В ДИРЕКТОРИИ DATA ---")
for filename in sorted(all_files):
    filepath = DATA_PATH + filename
    inspect_file(filepath, filename, n_rows=5)

# Анализируем  файлы из crimes
print("\n" + "="*50)
print("АНАЛИЗ ФАЙЛОВ ИЗ ПАПКИ CRIMES")
print("="*50)

# Берем по одному файлу из разных форматов
crime_samples = [
    'crimes/4-EGS_Razdel_4_012016.xls',  # старый формат, январь 2016
    'crimes/4-EGS_Razdel_4_122020.xls',  # старый формат, декабрь 2020
    'crimes/4-EGS_Razdel_5_012021.xls',  # новый формат, январь 2021
]

for crime_file in crime_samples:
    filepath = DATA_PATH + crime_file
    inspect_file(filepath, crime_file, n_rows=10)  # для crimes покажем больше строк


Всего файлов в data: 26

Список файлов:
  cash_real_income_wages_2015_2020.xlsx (32.8 КБ)
  child_mortality_rural_1990_2021.xls (89.0 КБ)
  child_mortality_urban_1990_2021.xls (89.0 КБ)
  crimes_2020_processed.csv (6.4 КБ)
  disabled_total_by_age_2017_2022.csv (577.6 КБ)
  drug_alco.xlsx (31.0 КБ)
  gross_regional_product_1996_2020.xls (62.0 КБ)
  housing_2020.xlsx (23.9 КБ)
  master_dataset_2020_clean.csv (18.6 КБ)
  master_dataset_2020_interim.csv (17.1 КБ)
  morbidity_2005_2020_age_disease.xls (2196.5 КБ)
  morbidity_2020_processed.csv (3.8 КБ)
  newborn_2006_2022_monthly.csv (102.8 КБ)
  newborn_2020_processed.csv (3.7 КБ)
  population.xlsx (52.4 КБ)
  poverty_percent_by_regions_1992_2020.csv (142.1 КБ)
  poverty_socdem_2017.xls (23.0 КБ)
  poverty_socdem_2018.xls (23.0 КБ)
  poverty_socdem_2019.xls (23.0 КБ)
  poverty_socdem_2020.xls (23.0 КБ)
  regional_production_2005_2016.csv (251.7 КБ)
  regional_production_2017_2020.csv (121.7 КБ)
  retail_turnover_per_capita_2000_2021.xls (

,0,1,2,3,4,5,6
0,region,2015,2016,2017,2018,2019,2020
1,Российская Федерация,302541,30865,31897,33266,35338,36073
2,Центральный\n федеральный округ,38832,40200,41897,44009,47252,48226
3,Белгородская область,28043,29799,30342,30778,32352,32841
4,Брянская область,23428,24006,25107,26585,28371,28596



Остальные листы: ['real_incomes', 'formal_wage_paid', 'real_pay', 'desc']

----------------------------------------
Файл: child_mortality_rural_1990_2021.xls
----------------------------------------
Тип: Excel
Листы: ['Данные', 'Паспорт']

Первый лист (первые 5 строк, без заголовков):


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34
0,Число умерших на первом году жизни детей за го...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,1990.00,1991.00,1992.00,1993.00,1994.00,1995.00,1996.00,1997.00,1998.00,1999.00,2000.00,2001.00,2002.00,2003.00,2004.00,2005.00,2006.00,2007.00,2008.00,2009.00,2010.00,2011.00,2012.00,2013.00,2014.00,2015.00,2016.00,2017.00,2018.00,2019.00,2020.00,2021.00
3,Российская Федерация,Оба пола,сельское население,11186.00,10943.00,10111.00,9840.00,9010.00,8582.00,7983.00,7701.00,7214.00,7074.00,6352.00,6205.00,5896.00,5907.00,5743.00,5357.00,5240.00,5361.00,5163.00,5082.00,4764.00,4770.00,5463.00,5059.00,4681.00,4165.00,3568.00,3032.00,2521.00,2122.00,1916.00,1852.00
4,Центральный федеральный округ,Оба пола,сельское население,1615.00,1550.00,1361.00,1426.00,1347.00,1321.00,1200.00,NaN,NaN,994.00,858.00,893.00,809.00,828.00,775.00,727.00,642.00,590.00,559.00,583.00,525.00,539.00,625.00,608.00,487.00,490.00,426.00,347.00,291.00,211.00,228.00,212.00



Остальные листы: ['Паспорт']

----------------------------------------
Файл: child_mortality_urban_1990_2021.xls
----------------------------------------
Тип: Excel
Листы: ['Данные', 'Паспорт']

Первый лист (первые 5 строк, без заголовков):


...


Остальные листы: ['Строка 2', 'Строка 3', 'Строка 4', 'Строка 5']

----------------------------------------
Файл: crimes/4-EGS_Razdel_4_122020.xls
----------------------------------------
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
Тип: Excel
Листы: ['Строка 1', 'Строка 2', 'Строка 3', 'Строка 4', 'Строка 5']
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero

Первый лист (первые 10 строк, без заголовков):


,0,1,2,3,4,5,6,7,8,9,10,11
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,ФОРМА 4-ЕГС,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ФОРМА 4-ЕГС
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,"Раздел 4. Сведения о преступлениях, совершенны...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,Строка 1: Всего,Неверное имя среза,Неверное имя среза,Неверное имя среза,Неверное имя среза,Неверное имя среза,Неверное имя среза,Неверное имя среза,Неверное имя среза,Неверное имя среза
6,NaN,NaN,Период: январь - декабрь 2020 года,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,Количество предварительно расследованных прест...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,несовершеннолетними или при их соучастии\n,ранее совершавшими преступления\n,в том числе,группой лиц\n,группой лиц по предварительному сговору\n,организованной группой\n,преступным сообществом (преступной организацие...,в состоянии опьянения,NaN,NaN



Остальные листы: ['Строка 2', 'Строка 3', 'Строка 4', 'Строка 5']

----------------------------------------
Файл: crimes/4-EGS_Razdel_5_012021.xls
----------------------------------------
Тип: Excel
Листы: ['Строка 1', 'Строка 2', 'Строка 3', 'Строка 4', 'Строка 5', 'Строка 6', 'Строка 7', 'Строка 8', 'Строка 9', 'Строка 10', 'Строка 11', 'Строка 12', 'Строка 13', 'Строка 14', 'Строка 15', 'Строка 16', 'Строка 17', 'Строка 18', 'Строка 19', 'Строка 20']

Первый лист (первые 10 строк, без заголовков):


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,ФОРМА 4-ЕГС,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ФОРМА 4-ЕГС,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,"Раздел 5. Сведения о преступлениях, совершенны...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,Строка 1: Всего,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,Период: январь 2021 года,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,Количество предварительно расследованных прест...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,несовершеннолетними или при их соучастии,ранее совершавшими преступления,в том числе,группой лиц,группой лиц по предварительному сговору,организованной группой,преступным сообществом (преступной организацией),в состоянии опьянения,NaN,NaN,"членом семьи, супругом, сожителем, сексуальным...",из них в отношении,NaN,иностранными гражданами и лицами без гражданства,из них,NaN,NaN



Остальные листы: ['Строка 2', 'Строка 3', 'Строка 4', 'Строка 5', 'Строка 6', 'Строка 7', 'Строка 8', 'Строка 9', 'Строка 10', 'Строка 11', 'Строка 12', 'Строка 13', 'Строка 14', 'Строка 15', 'Строка 16', 'Строка 17', 'Строка 18', 'Строка 19', 'Строка 20']


В результате первичного анализа структуры всех предоставленных файлов составлена сводная таблица:

| Файл | Формат | Период | Особенности |
|------|--------|--------|-------------|
| `cash_real_income_wages_2015_2020.xlsx` | Excel | 2015-2020 | 5 листов с разными показателями (доходы, зарплаты). Готов к использованию |
| `child_mortality_rural_1990_2021.xls` | Excel | 1990-2021 | Годы в строке 2, данные с 3 строки |
| `child_mortality_urban_1990_2021.xls` | Excel | 1990-2021 |Годы в строке 2, данные с 3 строки |
| `disabled_total_by_age_2017_2022.csv` | CSV | 2017-2022 | Готов к использованию |
| `drug_alco.xlsx` | Excel | 2005-2018 | Отдельные листы для алкоголизма и наркомании, за 2017-2018 данные в других листах |
| `gross_regional_product_1996_2020.xls` | Excel | 1996-2020 | Годы в строке 2, данные с 3 строки |
| `housing_2020.xlsx` | Excel | 2020 | Готов к использованию |
| `morbidity_2005_2020_age_disease.xls` | Excel | 2005-2016 | Заболеваемость с разбивкой по возрасту и болезням. Сложная структура |
| `newborn_2006_2022_monthly.csv` | CSV | 2006-2022 | Широкая таблица с помесячными данными. Требует преобразования |
| `population.xlsx` | Excel | 1999-2022 | Есть данные по муниципальным образованиям, требуется фильтрация |
| `poverty_percent_by_regions_1992_2020.csv` | CSV | 1992-2020 | Готов к использованию (tidy format) |
| `poverty_socdem_2017.xls` | Excel | 2017 | Готов к использованию |
| `poverty_socdem_2018.xls` | Excel | 2018 | Готов к использованию |
| `poverty_socdem_2019.xls` | Excel | 2019 | Готов к использованию |
| `poverty_socdem_2020.xls` | Excel | 2020 | Готов к использованию |
| `regional_production_2005_2016.csv` | CSV | 2005-2016 | Готов к использованию |
| `regional_production_2017_2020.csv` | CSV | 2017-2020 | Готов к использованию |
| `retail_turnover_per_capita_2000_2021.xls` | Excel | 2000-2021 | Годы в строке 2, данные с 4 строки |
| `welfare_expense_share_2015_2020.xlsx` | Excel | 2015-2020 | Готов к использованию |
| `workers.xls` | Excel | 2012-2016 | Данные до 2016г |
| `crimes/*.xls` | Excel | 2016-2022 | Требуется парсинг |

#### Основные наблюдения:

1. **Проблемные файлы**, требующие дополнительной обработки:
   - `morbidity_2005_2020_age_disease.xls` - сложная многоуровневая структура
   - `newborn_2006_2022_monthly.csv` - требует преобразования из широкого формата в длинный
   - Файлы crimes - требуют агрегации помесячных накопленных данных в годовые

2. **Проблема наименований регионов**: 
   - В разных файлах регионы могут называться по-разному (например, "г. Москва", "Москва", "Город Москва")
   - Тюменская и Архангельская области представлены как с учётом автономных округов, так и без
   - Присутствуют федеральные округа и РФ, которые нужно будет исключить из анализа.

#### 2.3 Подготовка данных к анализу

**Выбор репрезентативного года**

На основе анализа доступных данных (раздел 2.2) в качестве основного периода исследования выбран **2020 год** по следующим причинам:

- Присутствует в большинстве файлов (`poverty_percent`, `cash_real_income`, `disabled`, `housing`, `welfare`, `production`, `retail`, `grp`)
- Является последним годом с полным набором данных во многих источниках до начала методических изменений
- Доступна детальная структура малоимущего населения (`poverty_socdem_2020.xls`)
- При этом требует осторожной интерпретации из-за возможных аномалий, связанных с пандемией COVID-19

**Обработка проблемных файлов**

Перед формированием единого датасета необходимо привести к рабочему состоянию следующие файлы:
- `newborn_2006_2022_monthly.csv` - преобразование из широкого формата
- `workers.xls` - оценка возможности использования (отсутствие 2020)
- `morbidity_2005_2020_age_disease.xls` - анализ структуры
- `crimes/4-EGS_Razdel_4_122020.xls` - извлечение данных за 2020

In [3]:
# --- Функция нормализации наименований субъектов ---
def normalize_region(name):
    if pd.isna(name):
        return None
    
    name = str(name).strip()
    name_lower = name.lower()
    
    # Исключаем федеральные округа и РФ по ключевым словам
    if any(x in name_lower for x in ['федеральный округ', 'федерация', 'числе', 'байконур', 'долгано']):
        return None
    
    # Москва
    if 'москва' in name_lower and 'область' not in name_lower:
        return 'Москва'
    
    # Санкт-Петербург
    if 'санкт-петербург' in name_lower and 'область' not in name_lower:
        return 'Санкт-Петербург'
    
    # Севастополь
    if 'севастополь' in name_lower:
        return 'Севастополь'
    
    # Тюменская область
    if any(x in name_lower for x in ['тюменск', 'ханты-мансийск', 'ямало-ненецк']):
        return 'Тюменская область'

    # Архангельская область
    if any(x in name_lower for x in ['архангельск', 'ненецк']):
        return 'Архангельская область'
    
    # Специальные случаи республик
    if 'северная осетия' in name_lower:
        return 'Республика Северная Осетия-Алания'
    
    if 'адыгея' in name_lower:
        return 'Республика Адыгея'
    
    if 'кабардино-балкар' in name_lower:
        return 'Кабардино-Балкарская Республика'
    
    if 'карачаево-черкес' in name_lower:
        return 'Карачаево-Черкесская Республика'
    
    if 'ингушетия' in name_lower:
        return 'Республика Ингушетия'
    
    if 'чечен' in name_lower:
        return 'Чеченская Республика'
    
    if 'чуваш' in name_lower:
        return 'Чувашская Республика'
    
    if 'удмурт' in name_lower:
        return 'Удмуртская Республика'
    
    # Обработка республик общего вида
    if name_lower.startswith('республика') or 'республика' in name_lower:
        # Очищаем от скобок и дополнительных пометок
        name = re.sub(r'\s*\([^)]*\)', '', name)
        name = re.sub(r'\s*\[[^\]]*\]', '', name)
        name = re.sub(r'\s*—.*$', '', name)
        name = re.sub(r'\s*\-.*$', '', name)
        # Убираем лишние пробелы
        name = ' '.join(name.split())
        return name
    
    # Кемеровская область
    if 'кемеровская' in name_lower:
        return 'Кемеровская область'
    
    return name

In [4]:
# --- Обработка данных о рождаемости (newborn_2006_2022_monthly.csv) ---
output_file = DATA_PATH + "newborn_2020_processed.csv"

# Удаление выходного файла, если он существует
if os.path.exists(output_file):
    os.remove(output_file)

# 1. Загрузка
df = pd.read_csv(DATA_PATH + 'newborn_2006_2022_monthly.csv', sep=';', encoding='utf-8')
df = df.rename(columns={df.columns[0]: 'region'})
if 'Unnamed: 198' in df.columns:
    df = df.drop(columns=['Unnamed: 198'])

# 2. Преобразование в длинный формат
df_long = pd.melt(df, id_vars=['region'], var_name='month_year', value_name='births')

# 3. Очистка значений от артефактов ",99"
df_long['births'] = df_long['births'].astype(str).str.replace(',99', '', regex=False)
df_long['births'] = df_long['births'].str.replace(' ', '', regex=False)
df_long['births'] = pd.to_numeric(df_long['births'], errors='coerce')
df_long = df_long.dropna(subset=['births'])

# 4. Извлечение года и месяца
months = {
    'январь': 1, 'февраль': 2, 'март': 3, 'апрель': 4,
    'май': 5, 'июнь': 6, 'июль': 7, 'август': 8,
    'сентябрь': 9, 'октябрь': 10, 'ноябрь': 11, 'декабрь': 12
}

def extract_year_month(date_str):
    for month_name, month_num in months.items():
        if month_name in date_str:
            year_match = re.search(r'(\d{4})', date_str)
            if year_match:
                return int(year_match.group(1)), month_num
    return None, None

df_long[['year', 'month']] = df_long['month_year'].apply(lambda x: pd.Series(extract_year_month(x)))

# 5. Фильтр за 2020 год и агрегация по регионам
df_2020 = df_long[df_long['year'] == 2020]
df_agg = df_2020.groupby('region')['births'].sum().reset_index()
df_agg = df_agg.rename(columns={'births': 'births_2020'})

# 6. Нормализация названий регионов
df_agg['region'] = df_agg['region'].apply(normalize_region)
df_agg = df_agg.dropna(subset=['region']).reset_index(drop=True)
# дополнительная агрегация для объединения дубликатов
df_agg = df_agg.groupby('region')['births_2020'].sum().reset_index()

# 7. Сохранение результата
df_agg.to_csv(output_file, index=False)

print(f"Обработано {len(df_agg)} регионов")
print(f"Файл сохранён: {output_file}")
print("Пример данных:")
display(df_agg.head())

Обработано 82 регионов
Файл сохранён: data/newborn_2020_processed.csv
Пример данных:


,region,births_2020
0,Алтайский край,19976.00
1,Амурская область,7853.00
2,Архангельская область,19000.00
3,Астраханская область,10910.00
4,Белгородская область,12351.00


In [5]:
### Обработка workers.xls
from sklearn.linear_model import LinearRegression
output_file = DATA_PATH + "workers_processed.csv"

# Удаляем существующий файл, если он есть
if os.path.exists(output_file):
    os.remove(output_file)

# 1. Загрузка
df_raw = pd.read_excel(DATA_PATH + 'workers.xls', sheet_name=0, header=None)

# Строка с индексом 2 содержит года
years = df_raw.iloc[2, 2:].values

# Строки с 4-й (индекс 4) содержат данные
regions = df_raw.iloc[4:, 1].values  # Названия регионов
values = df_raw.iloc[4:, 2:].values  # Значения

# Создаём DataFrame
df = pd.DataFrame(values, columns=[str(int(y)) for y in years])
df.insert(0, 'region', regions)

# 2. Преобразование в длинный формат
df_long = pd.melt(df, id_vars=['region'], var_name='year', value_name='workers_percent')
df_long['year'] = pd.to_numeric(df_long['year'])
df_long = df_long.dropna(subset=['workers_percent']).reset_index(drop=True)

# 3. Нормализация названий регионов
df_long['region'] = df_long['region'].apply(normalize_region)
df_clean = df_long.dropna(subset=['region']).copy()
df_clean = df_clean.groupby(['region', 'year'], as_index=False)['workers_percent'].mean()

print(f"Уникальных регионов после нормализации: {df_clean['region'].nunique()}")

# 4. Прогнозирование на 2020 год
regions = df_clean['region'].unique()
predicted_data = []

for region in regions:
    region_data = df_clean[df_clean['region'] == region].copy()
    region_data = region_data.sort_values('year')
    
    if len(region_data) < 2:
        # Если данных мало, используем последнее значение
        last_value = region_data['workers_percent'].iloc[-1]
        predicted_data.append({
            'region': region,
            'year': 2020,
            'workers_percent': last_value
        })
        continue
    
    X = region_data['year'].values.reshape(-1, 1)
    y = region_data['workers_percent'].values
    
    model = LinearRegression()
    model.fit(X, y)
    
    pred_2020 = model.predict([[2020]])[0]
    pred_2020 = max(0, min(100, pred_2020))  # Ограничиваем 0-100%
    
    predicted_data.append({
        'region': region,
        'year': 2020,
        'workers_percent': round(pred_2020, 2)
    })

df_pred = pd.DataFrame(predicted_data)

# 5. Сохранение результата
df_pred.to_csv(output_file, index=False, encoding='utf-8')

print(f"Файл сохранён: {output_file}")
print("Пример данных:")
display(df_pred.head())

Уникальных регионов после нормализации: 82
Файл сохранён: data/workers_processed.csv
Пример данных:


,region,year,workers_percent
0,Алтайский край,2020,81.74
1,Амурская область,2020,85.54
2,Архангельская область,2020,100.00
3,Астраханская область,2020,93.44
4,Белгородская область,2020,95.92


In [6]:
# --- Обработка заболеваемости (morbidity_2005_2020_age_disease.xls) ---
output_file = DATA_PATH + "morbidity_2020_processed.csv"

if os.path.exists(output_file):
    os.remove(output_file)

# 1. Загрузка с правильной строкой заголовков
df_morb = pd.read_excel(
    DATA_PATH + 'morbidity_2005_2020_age_disease.xls',
    sheet_name=0,
    header=2
)

df_morb = df_morb.rename(columns={
    'Unnamed: 0': 'region',
    'Unnamed: 1': 'disease',
    'Unnamed: 2': 'age_group'
})

# 2. Фильтрация итоговых строк
df_total = df_morb[
    (df_morb['disease'] == 'Все заболевания') & 
    (df_morb['age_group'] == 'Всего')
].copy()

# 3. Преобразование в длинный формат (сразу после фильтрации)
years = [str(y) for y in range(2005, 2017)]
df_long = pd.melt(
    df_total,
    id_vars=['region', 'disease', 'age_group'],
    value_vars=years,
    var_name='year',
    value_name='morbidity_rate'
)

# Удаляем служебные колонки
df_long = df_long.drop(columns=['disease', 'age_group'])
df_long['year'] = pd.to_numeric(df_long['year'])
df_long = df_long.dropna(subset=['morbidity_rate']).reset_index(drop=True)

# 4. Нормализация названий регионов (федеральные округа отсеются автоматически)
df_long['region'] = df_long['region'].apply(normalize_region)
df_clean = df_long.dropna(subset=['region']).copy()
df_clean = df_clean.groupby(['region', 'year'], as_index=False)['morbidity_rate'].mean()

# 5. Прогнозирование на 2020
from sklearn.linear_model import LinearRegression

regions = df_clean['region'].unique()
predicted_data = []

for region in regions:
    region_data = df_clean[df_clean['region'] == region].copy()
    region_data = region_data.sort_values('year')
    
    if len(region_data) < 2:
        last_value = region_data['morbidity_rate'].iloc[-1]
        predicted_data.append({
            'region': region,
            'morbidity_rate_2020': last_value
        })
        continue
    
    X = region_data['year'].values.reshape(-1, 1)
    y = region_data['morbidity_rate'].values
    
    model = LinearRegression()
    model.fit(X, y)
    
    pred_2020 = model.predict([[2020]])[0]
    pred_2020 = max(0, round(pred_2020, 2))
    
    predicted_data.append({
        'region': region,
        'morbidity_rate_2020': pred_2020
    })

df_pred = pd.DataFrame(predicted_data)
df_pred.to_csv(output_file, index=False, encoding='utf-8')

print(f"Обработано {len(df_pred)} регионов")
print(f"Файл сохранён: {output_file}")
print("Пример данных:")
display(df_pred.head())

Обработано 82 регионов
Файл сохранён: data/morbidity_2020_processed.csv
Пример данных:


,region,morbidity_rate_2020
0,Алтайский край,116618.39
1,Амурская область,93183.50
2,Архангельская область,123076.22
3,Астраханская область,62153.14
4,Белгородская область,72498.65


In [7]:
# --- Обработка данных о преступности (crimes) ---
# Извлечение данных за 2020 год из файла 4-EGS_Razdel_4_122020.xls

output_file = "data/crimes_2020_processed.csv"

# Удаление выходного файла, если он существует
if os.path.exists(output_file):
    os.remove(output_file)

# 1. Загрузка файла
file_path = DATA_PATH + 'crimes/4-EGS_Razdel_4_122020.xls'
xl = pd.ExcelFile(file_path)

# Используем лист 'Строка 1' (основные преступления)
df = pd.read_excel(file_path, sheet_name='Строка 1', header=None)

# 2. Извлечение названий категорий
row_9 = df.iloc[9].fillna('').astype(str).tolist()
row_10 = df.iloc[10].fillna('').astype(str).tolist()

# Формируем названия колонок
column_names = ['drop', 'region',
                'несовершеннолетние',
                'ранее_совершавшие',
                'ранее_судимые',
                'группа_лиц',
                'группа_лиц_предварительный_сговор',
                'организованная_группа',
                'преступное_сообщество',
                'в_состоянии_опьянения_всего',
                'алкогольное_опьянение',
                'наркотическое_опьянение']

# 3. Создание DataFrame с данными
df_crimes = df.iloc[12:].copy()
df_crimes.columns = column_names
df_crimes = df_crimes.drop(columns=['drop'])

# Преобразование в числовой формат
def to_numeric(x):
    if pd.isna(x) or x == '' or x == ' ':
        return 0
    try:
        return float(str(x).replace(' ', '').replace(',', '.'))
    except:
        return 0

numeric_columns = [col for col in df_crimes.columns if col != 'region']
for col in numeric_columns:
    df_crimes[col] = df_crimes[col].apply(to_numeric).astype(int)

# 4. Исключаем транспортную полицию и артефакты (это не регионы)
df_regions = df_crimes[~df_crimes['region'].str.contains('УТ МВД|Транспорт', na=False, case=False)].copy()

# 5. Нормализация названий регионов (используем общую функцию)
df_regions['region'] = df_regions['region'].apply(normalize_region)
df_clean = df_regions.dropna(subset=['region']).copy()

# 6. Агрегация данных по очищенным названиям
df_agg = df_clean.groupby('region')[numeric_columns].sum().reset_index()

print(f"Получено регионов: {len(df_agg)}")

# 7. Сохранение результата
df_agg.to_csv(output_file, index=False, encoding='utf-8')
print(f"Файл сохранён: {output_file}")
print("Пример данных:")
print(df_agg.head()[['region']])

WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
Получено регионов: 82
Файл сохранён: data/crimes_2020_processed.csv
Пример данных:
                  region
0         Алтайский край
1       Амурская область
2  Архангельская область
3   Астраханская область
4   Белгородская область


#### 2.4 Итоговый датасет: структура и статистики
- Сбор всех показателей в один DataFrame
- Проверка полноты данных

После обработки проблемных файлов мы приступаем к сборке итоговой таблицы. Все показатели приводятся к единому формату с использованием стандартизированных названий регионов. В качестве целевого года определён **2020**.

##### Целевая переменная

| Показатель | Описание | Источник |
|------------|----------|----------|
| `poverty_percent` | Доля населения с доходами ниже прожиточного минимума, % | `poverty_percent_by_regions_1992_2020.csv` |

##### Факторные признаки

**Экономические показатели**

| Показатель | Описание | Источник |
|------------|----------|----------|
| `income_per_capita` | Среднедушевые денежные доходы, руб. | `cash_real_income_wages_2015_2020.xlsx`, лист `per_capita_cash_income` |
| `grp_per_capita` | ВРП на душу населения, руб. | `gross_regional_product_1996_2020.xls` |
| `retail_turnover_per_capita` | Оборот розничной торговли на душу, руб. | `retail_turnover_per_capita_2000_2021.xls` |
| `workers_percent` | Доля занятых в трудоспособном населении, % | `workers_processed.csv` (прогноз на 2020) |

**Демография и социальная защита**

| Показатель | Описание | Источник |
|------------|----------|----------|
| `births_2020` | Число родившихся за 2020 год, чел. | `newborn_2020_processed.csv` |
| `disabled_total` | Общая численность инвалидов, чел. | `disabled_total_by_age_2017_2022.csv` (данные на 01.01.2020) |

**Здравоохранение и преступность**

| Показатель | Описание | Источник |
|------------|----------|----------|
| `morbidity_rate` | Заболеваемость на 100 тыс. чел. (все болезни) | `morbidity_2020_processed.csv` |
| `crimes_juveniles` | Преступления, совершенные несовершеннолетними | `crimes_2020_processed.csv` |
| `crimes_alcohol` | Преступления, совершенные в состоянии опьянения | `crimes_2020_processed.csv` |
| `crimes_repeat` | Преступления, совершенные ранее совершавшими | `crimes_2020_processed.csv` |

**Жилищные условия и бюджет**

| Показатель | Описание | Источник |
|------------|----------|----------|
| `housing_satisfied` | Доля домохозяйств, удовлетворенных жильем, % | `housing_2020.xlsx` |
| `housing_unsatisfied` | Доля домохозяйств, не удовлетворенных жильем, % | `housing_2020.xlsx` |
| `housing_area_per_person` | Общая площадь на человека, кв. м | `housing_2020.xlsx` |
| `welfare_expense_share` | Доля расходов на соцполитику в бюджете региона, % | `welfare_expense_share_2015_2020.xlsx` |

##### Структура малоимущего населения (для отдельного анализа)

Для описания уязвимых групп используются файлы `poverty_socdem_2020.xls`, содержащие распределение малоимущего населения по возрастным группам (дети, трудоспособные, пенсионеры).

In [8]:
# --- Загрузка целевой переменной: уровень бедности ---

# Загрузка
df_poverty = pd.read_csv(
    DATA_PATH + 'poverty_percent_by_regions_1992_2020.csv',
    encoding='utf-8'
)

# Фильтр на 2020 год
df_poverty_2020 = df_poverty[df_poverty['year'] == 2020].copy()

# Нормализация регионов
df_poverty_2020['region'] = df_poverty_2020['region'].apply(normalize_region)
df_poverty_2020 = df_poverty_2020.groupby('region', as_index=False)['poverty_percent'].mean()

# Удаляем нераспознанные регионы (None)
df_poverty_2020 = df_poverty_2020.dropna(subset=['region']).reset_index(drop=True)

# Оставляем только нужные колонки
df_poverty_2020 = df_poverty_2020[['region', 'poverty_percent']]

# Проверка
print(f"Получено регионов: {len(df_poverty_2020)}")
print(df_poverty_2020.head())

Получено регионов: 82
                  region  poverty_percent
0         Алтайский край            17.50
1       Амурская область            15.20
2  Архангельская область            11.50
3   Астраханская область            15.60
4   Белгородская область             7.20


In [9]:
# --- Инициализация единого датасета ---
df_agg = df_poverty_2020.copy()

In [10]:
# --- Загрузка: среднедушевые доходы ---

# Загрузка (лист per_capita_cash_income, первая строка — заголовок)
df_income = pd.read_excel(
    DATA_PATH + 'cash_real_income_wages_2015_2020.xlsx',
    sheet_name='per_capita_cash_income',
    header=0
)

# Переименуем первую колонку в 'region'
df_income = df_income.rename(columns={df_income.columns[0]: 'region'})

# Нормализация
df_income['region'] = df_income['region'].apply(normalize_region)
df_income = df_income.dropna(subset=['region']).reset_index(drop=True)

# Оставляем регион и 2020 год (как число)
df_income_2020 = df_income[['region', 2020]].copy()
df_income_2020 = df_income_2020.rename(columns={2020: 'income_per_capita'})
df_income_2020 = df_income_2020.groupby('region', as_index=False)['income_per_capita'].mean()

print(f"Получено регионов: {len(df_income_2020)}")

df_agg = df_agg.merge(df_income_2020, on='region', how='left')

print(f"Размер после слияния: {df_agg.shape}")
print(f"Колонки: {df_agg.columns.tolist()}")
print(df_agg.head(3))

Получено регионов: 82
Размер после слияния: (82, 3)
Колонки: ['region', 'poverty_percent', 'income_per_capita']
                  region  poverty_percent  income_per_capita
0         Алтайский край            17.50           23864.00
1       Амурская область            15.20           35499.00
2  Архангельская область            11.50           51926.00


In [11]:
# --- Загрузка: ВРП на душу населения ---
print("Загрузка gross_regional_product...")

# Загрузка
df_grp = pd.read_excel(
    DATA_PATH + 'gross_regional_product_1996_2020.xls',
    sheet_name='Данные',
    header=2
)

# Переименуем первую колонку в 'region'
df_grp = df_grp.rename(columns={df_grp.columns[0]: 'region'})

# Нормализация регионов
df_grp['region'] = df_grp['region'].apply(normalize_region)

# Удаляем нераспознанные регионы (None)
df_grp = df_grp.dropna(subset=['region']).reset_index(drop=True)

# Оставляем регион и 2020 год
df_grp_2020 = df_grp[['region', '2020']].copy()

# Группируем на случай дублей
df_grp_2020 = df_grp_2020.groupby('region', as_index=False)['2020'].mean()
df_grp_2020 = df_grp_2020.rename(columns={'2020': 'grp_per_capita'})

print(f"Получено регионов: {len(df_grp_2020)}")

# Добавляем в единый датасет
df_agg = df_agg.merge(df_grp_2020, on='region', how='left')
print(f"Размер df_agg: {df_agg.shape}")
print(df_agg.head(3))

Загрузка gross_regional_product...
Получено регионов: 82
Размер df_agg: (82, 4)
                  region  poverty_percent  income_per_capita  grp_per_capita
0         Алтайский край            17.50           23864.00       291156.90
1       Амурская область            15.20           35499.00       571362.10
2  Архангельская область            11.50           51926.00      2139378.57


In [12]:
# --- Загрузка: оборот розничной торговли на душу ---

# Загружаем без заголовков
df_retail_raw = pd.read_excel(
    DATA_PATH + 'retail_turnover_per_capita_2000_2021.xls',
    sheet_name='Данные',
    header=None
)

# Строка 3 содержит годы, начиная с колонки 2
years = df_retail_raw.iloc[2, 2:].astype(str).tolist()

# Данные начинаются со строки 5
df_retail = df_retail_raw.iloc[4:].copy()

# Первая колонка — регион, вторая — единица измерения (рубль), удаляем её
df_retail = df_retail.drop(columns=[1])

# Назначаем имена колонок
df_retail.columns = ['region'] + years

# Нормализация
df_retail['region'] = df_retail['region'].apply(normalize_region)
df_retail = df_retail.dropna(subset=['region']).reset_index(drop=True)
df_retail = df_retail.groupby('region', as_index=False)[years].mean()

for year in years:
    df_retail[year] = pd.to_numeric(df_retail[year], errors='coerce')

# Берём 2020 год
df_retail_2020 = df_retail[['region', '2020']].copy()
df_retail_2020 = df_retail_2020.rename(columns={'2020': 'retail_turnover_per_capita'})

# Добавляем в единый датасет
df_agg = df_agg.merge(df_retail_2020, on='region', how='left')

print(f"Получено регионов: {len(df_retail_2020)}")
print(df_retail_2020.head())
print(f"Размер df_agg: {df_agg.shape}")

Получено регионов: 87
                                          region  retail_turnover_per_capita
0  Агинский Бурятский округ (Забайкальский край)                         NaN
1                                 Алтайский край                   153605.00
2                               Амурская область                   245233.00
3                          Архангельская область                   241780.00
4                           Астраханская область                   174527.00
Размер df_agg: (82, 5)


In [13]:
# --- Загрузка: доля занятых (workers) ---

df_workers = pd.read_csv(
    DATA_PATH + 'workers_processed.csv',
    encoding='utf-8'
)

print(f"Получено регионов: {len(df_workers)}")
print(df_workers.head())

# Добавляем в df_agg
df_agg = df_agg.merge(df_workers, on='region', how='left')
print(f"Размер df_agg: {df_agg.shape}")

Получено регионов: 82
                  region  year  workers_percent
0         Алтайский край  2020            81.74
1       Амурская область  2020            85.54
2  Архангельская область  2020           100.00
3   Астраханская область  2020            93.44
4   Белгородская область  2020            95.92
Размер df_agg: (82, 7)


In [14]:
# --- Загрузка: численность инвалидов (disabled) ---

df_disabled = pd.read_csv(
    DATA_PATH + 'disabled_total_by_age_2017_2022.csv',
    encoding='utf-8'
)

# Нормализация
df_disabled['region'] = df_disabled['region'].apply(normalize_region)
df_disabled = df_disabled.dropna(subset=['region']).reset_index(drop=True)
df_disabled = df_disabled.groupby(['region', 'date'], as_index=False)['total'].mean()

# Преобразуем дату и берём ближайшую к 2020
df_disabled['date'] = pd.to_datetime(df_disabled['date'])
df_2020 = df_disabled[df_disabled['date'] == '2020-01-01'].copy()

# Если нет точной даты, берём максимум (позднюю дату до 2020)
if len(df_2020) == 0:
    df_disabled = df_disabled[df_disabled['date'] < '2020-01-01']
    df_2020 = df_disabled.loc[df_disabled.groupby('region')['date'].idxmax()].reset_index(drop=True)

df_disabled_2020 = df_2020[['region', 'total']].copy()
df_disabled_2020 = df_disabled_2020.rename(columns={'total': 'disabled_total'})

print(f"Получено регионов: {len(df_disabled_2020)}")
print(df_disabled_2020.head())

# Добавляем в df_agg
df_agg = df_agg.merge(df_disabled_2020, on='region', how='left')
print(f"Размер df_agg: {df_agg.shape}")

Получено регионов: 82
                    region  disabled_total
36          Алтайский край       173935.00
100       Амурская область        64692.00
164  Архангельская область        44243.50
228   Астраханская область        43609.00
292   Белгородская область       199929.00
Размер df_agg: (82, 8)


In [15]:
# --- Загрузка: число родившихся (births) ---

df_births = pd.read_csv(
    DATA_PATH + 'newborn_2020_processed.csv',
    encoding='utf-8'
)

# Нормализация
df_births['region'] = df_births['region'].apply(normalize_region)
df_births = df_births.dropna(subset=['region']).reset_index(drop=True)

# Переименовываем колонку
df_births = df_births.rename(columns={'births_2020': 'births'})

print(f"Получено регионов: {len(df_births)}")
print(df_births.head())

# Добавляем в df_agg
df_agg = df_agg.merge(df_births, on='region', how='left')
print(f"Размер df_agg: {df_agg.shape}")

Получено регионов: 82
                  region   births
0         Алтайский край 19976.00
1       Амурская область  7853.00
2  Архангельская область 19000.00
3   Астраханская область 10910.00
4   Белгородская область 12351.00
Размер df_agg: (82, 9)


In [16]:
# --- Загрузка: заболеваемость (morbidity) ---

df_morb = pd.read_csv(
    DATA_PATH + 'morbidity_2020_processed.csv',
    encoding='utf-8'
)

print(f"Получено регионов: {len(df_morb)}")
print(df_morb.head())

# Добавляем в df_agg
df_agg = df_agg.merge(df_morb, on='region', how='left')
print(f"Размер df_agg: {df_agg.shape}")

Получено регионов: 82
                  region  morbidity_rate_2020
0         Алтайский край            116618.39
1       Амурская область             93183.50
2  Архангельская область            123076.22
3   Астраханская область             62153.14
4   Белгородская область             72498.65
Размер df_agg: (82, 10)


In [17]:
# --- Загрузка: преступность (crimes) ---

df_crimes = pd.read_csv(
    DATA_PATH + 'crimes_2020_processed.csv',
    encoding='utf-8'
)

# Нормализация
df_crimes['region'] = df_crimes['region'].apply(normalize_region)
df_crimes = df_crimes.dropna(subset=['region']).reset_index(drop=True)

print(f"Получено регионов: {len(df_crimes)}")
print(df_crimes.head())

# Добавляем в df_agg
df_agg = df_agg.merge(df_crimes, on='region', how='left')
print(f"Размер df_agg: {df_agg.shape}")

Получено регионов: 82
                  region  несовершеннолетние  ранее_совершавшие  ранее_судимые  группа_лиц  группа_лиц_предварительный_сговор  организованная_группа  преступное_сообщество  в_состоянии_опьянения_всего  алкогольное_опьянение  наркотическое_опьянение
0         Алтайский край                 887              15400           7278          64                                901                     87                      2                         9396                     46                        0
1       Амурская область                 431               6001           3078          52                                409                     57                      0                         3123                    108                        2
2  Архангельская область                 571               6818           3648          49                                994                     54                      0                         4222                     32        

In [18]:
# --- Загрузка: жилищные условия (housing) ---
print("Загрузка housing...")

df_housing = pd.read_excel(
    DATA_PATH + 'housing_2020.xlsx',
    sheet_name='housing_cond',
    header=0
)

df_housing = df_housing.rename(columns={df_housing.columns[0]: 'region'})

# Нормализация
df_housing['region'] = df_housing['region'].apply(normalize_region)
df_housing = df_housing.dropna(subset=['region']).reset_index(drop=True)
df_housing = df_housing.groupby('region', as_index=False).mean()

# Выбираем и переименовываем нужные колонки
df_housing_2020 = df_housing[[
    'region',
    'в том числе домохозяйства, указавшие, что при проживании не испытывают стесненности',
    'Размер общей площади в расчете на члена домохозяйства',
    'Размер жилой площади в расчете на члена домохозяйства',
    'Число жилых комнат в расчете на одно домохозяйство'
]].copy()

df_housing_2020 = df_housing_2020.rename(columns={
    'в том числе домохозяйства, указавшие, что при проживании не испытывают стесненности': 'housing_satisfied_pct',
    'Размер общей площади в расчете на члена домохозяйства': 'housing_area_total_per_person',
    'Размер жилой площади в расчете на члена домохозяйства': 'housing_area_living_per_person',
    'Число жилых комнат в расчете на одно домохозяйство': 'housing_rooms_per_household'
})

print(f"Получено регионов: {len(df_housing_2020)}")
print(df_housing_2020.head())

# Добавляем в df_agg
df_agg = df_agg.merge(df_housing_2020, on='region', how='left')
print(f"Размер df_agg: {df_agg.shape}")

Загрузка housing...
Получено регионов: 82
                  region  housing_satisfied_pct  housing_area_total_per_person  housing_area_living_per_person  housing_rooms_per_household
0   Bладимирская область                  81.80                          24.20                           16.10                         2.20
1  Bолгоградская область                  82.70                          25.60                           17.90                         2.50
2    Bологодская область                  87.50                          22.90                           16.90                         2.20
3    Bоронежская область                  87.00                          29.00                           19.90                         2.60
4         Алтайский край                  83.20                          25.90                           18.00                         2.40
Размер df_agg: (82, 24)


In [19]:
# --- Загрузка: доля расходов на соцполитику (welfare) ---

df_welfare = pd.read_excel(
    DATA_PATH + 'welfare_expense_share_2015_2020.xlsx',
    sheet_name='data',
    header=0
)

# Переименуем первую колонку в 'region'
df_welfare = df_welfare.rename(columns={df_welfare.columns[0]: 'region'})

# Нормализация
df_welfare['region'] = df_welfare['region'].apply(normalize_region)
df_welfare = df_welfare.dropna(subset=['region']).reset_index(drop=True)

# Оставляем регион и 2020 год
df_welfare_2020 = df_welfare[['region', 2020]].copy()
df_welfare_2020 = df_welfare_2020.rename(columns={2020: 'welfare_expense_share'})
df_welfare_2020 = df_welfare_2020.groupby('region', as_index=False)['welfare_expense_share'].mean()

print(f"Получено регионов: {len(df_welfare_2020)}")
print(df_welfare_2020.head())

# Добавляем в df_agg
df_agg = df_agg.merge(df_welfare_2020, on='region', how='left')
print(f"Размер df_agg: {df_agg.shape}")

Получено регионов: 82
                  region  welfare_expense_share
0         Алтайский край                  29.70
1       Амурская область                  21.70
2  Архангельская область                  17.90
3   Астраханская область                  31.40
4   Белгородская область                  16.10
Размер df_agg: (82, 25)


In [20]:
# --- Загрузка: структура малоимущего населения по возрасту (poverty_socdem_2020) ---

df_socdem = pd.read_excel(
    DATA_PATH + 'poverty_socdem_2020.xls',
    sheet_name='Данные',
    header=None
)

# Пропускаем первые 2 строки (индексы 0-1), строка 2 (индекс 2) — заголовки
df_socdem.columns = df_socdem.iloc[2].fillna('').astype(str).tolist()
df_socdem = df_socdem.iloc[3:].copy()

# Переименовываем первую колонку в 'region' и очищаем
df_socdem = df_socdem.rename(columns={df_socdem.columns[0]: 'region'})
df_socdem = df_socdem[['region', 'Все население', 'Дети в возрасте до 16 лет', 
                       'Население трудоспособного возраста', 'Население старше трудоспособного возраста']].copy()

# Переименовываем колонки для удобства
df_socdem = df_socdem.rename(columns={
    'Все население': 'poor_total_pct',
    'Дети в возрасте до 16 лет': 'poor_children_pct',
    'Население трудоспособного возраста': 'poor_working_pct',
    'Население старше трудоспособного возраста': 'poor_pensioners_pct'
})

# Нормализация регионов
df_socdem['region'] = df_socdem['region'].apply(normalize_region)
df_socdem = df_socdem.dropna(subset=['region']).reset_index(drop=True)

# Преобразование в числовой формат
for col in ['poor_total_pct', 'poor_children_pct', 'poor_working_pct', 'poor_pensioners_pct']:
    df_socdem[col] = pd.to_numeric(df_socdem[col], errors='coerce')

# Группировка на случай дубликатов
df_socdem = df_socdem.groupby('region', as_index=False).mean()

print(f"Получено регионов: {len(df_socdem)}")
print(df_socdem.head())

# Добавляем в df_agg
df_agg = df_agg.merge(df_socdem, on='region', how='left')
print(f"Размер df_agg: {df_agg.shape}")

Получено регионов: 82
                  region  poor_total_pct  poor_children_pct  poor_working_pct  poor_pensioners_pct
0         Алтайский край          100.00              31.40             59.40                 9.20
1       Амурская область          100.00              38.40             56.90                 4.70
2  Архангельская область          100.00              29.35             63.70                 6.95
3   Астраханская область          100.00              42.10             52.40                 5.50
4   Белгородская область          100.00                NaN               NaN                  NaN
Размер df_agg: (82, 29)


In [21]:
# --- Загрузка: численность населения (population) с учётом структуры ---

# Загружаем без заголовков
df_pop_raw = pd.read_excel(
    DATA_PATH + 'population.xlsx',
    sheet_name='Отчет',
    header=None
)

# Строка 1 (индекс 1) содержит названия годов
years = df_pop_raw.iloc[1, 2:].astype(str).tolist()

# Создаём списки для сбора данных
regions = []
populations = []

# Проходим по строкам, начиная с 4-й (индекс 4)
i = 4
while i < len(df_pop_raw):
    region_name = df_pop_raw.iloc[i, 0]  # Название региона в колонке 0
    
    # Проверяем, что это не пустая строка и не раздел
    if pd.notna(region_name) and not str(region_name).startswith('Раздел'):
        # Данные населения находятся в следующей строке (i+1)
        if i + 1 < len(df_pop_raw):
            pop_values = df_pop_raw.iloc[i + 1, 2:].values  # Берём данные с колонки 2
            
            # Добавляем в списки
            regions.append(region_name)
            populations.append(pop_values)
            
            # Пропускаем строку с данными, переходим к следующему региону
            i += 2
        else:
            i += 1
    else:
        i += 1

# Создаём DataFrame
df_pop = pd.DataFrame(populations, columns=years)
df_pop.insert(0, 'region', regions)

# Нормализация регионов
df_pop['region'] = df_pop['region'].apply(normalize_region)
df_pop = df_pop.dropna(subset=['region']).reset_index(drop=True)

# Преобразование в числовой формат
for col in years:
    df_pop[col] = pd.to_numeric(df_pop[col], errors='coerce')

# Группировка для объединения дубликатов (Тюменская и Астраханская области с округами)
df_pop = df_pop.groupby('region', as_index=False)[years].mean()

# Берём 2020 год (январь 2020 г.)
df_pop_2020 = df_pop[['region', 'январь 2020 г.']].copy()
df_pop_2020 = df_pop_2020.rename(columns={'январь 2020 г.': 'population_2020'})

print(f"Получено регионов после нормализации: {len(df_pop_2020)}")
print(df_pop_2020.head(10))

# Добавляем в df_agg. Добавятся только совпадающие, сельсоветы не войдут
df_agg = df_agg.merge(df_pop_2020, on='region', how='left')
print(f"Размер df_agg: {df_agg.shape}")

Получено регионов после нормализации: 162
                                          region  population_2020
0  Агинский Бурятский округ (Забайкальский край)              NaN
1                               Айский сельсовет         14228.00
2                         Акутихинский сельсовет          3189.00
3                          Аламбайский сельсовет           675.00
4                             Алейский сельсовет           550.00
5                                 Алтайский край       2317153.00
6                               Амурская область        790044.00
7                          Архангельская область        757690.00
8                           Астраханская область       1005782.00
9                   Баевский муниципальный район          3984.00
Размер df_agg: (82, 30)


In [22]:
# --- Загрузка и объединение данных о младенческой смертности (исправленная версия) ---

# Загружаем сельскую смертность
df_child_mort_rural = pd.read_excel(
    DATA_PATH + 'child_mortality_rural_1990_2021.xls',
    sheet_name='Данные',
    header=None
)

# Загружаем городскую смертность
df_child_mort_urban = pd.read_excel(
    DATA_PATH + 'child_mortality_urban_1990_2021.xls',
    sheet_name='Данные',
    header=None
)

def process_child_mortality_fixed(df):
    """функция обработки файлов младенческой смертности"""
    
    # Строка 3 содержит годы, начиная с колонки 3
    years_row = df.iloc[2, 3:].values
    years = [str(int(year)) for year in years_row if pd.notna(year)]
    
    # Создаем список для данных
    data_rows = []
    
    # Данные начинаются со строки 4
    for i in range(3, len(df)):
        region_name = df.iloc[i, 0]
        
        # Пропускаем пустые строки и итоговые строки
        if pd.notna(region_name) and not 'федеральный округ' in str(region_name).lower() and not 'российская федерация' in str(region_name).lower():
            values = df.iloc[i, 3:3+len(years)].values
            data_rows.append([region_name] + list(values))
    
    # Создаем DataFrame
    result_df = pd.DataFrame(data_rows, columns=['region'] + years)
    
    # Нормализация регионов
    result_df['region'] = result_df['region'].apply(normalize_region)
    result_df = result_df.dropna(subset=['region']).reset_index(drop=True)
    
    # Преобразование в числовой формат
    for col in years:
        result_df[col] = pd.to_numeric(result_df[col], errors='coerce')
    
    # Группировка для объединения дубликатов
    result_df = result_df.groupby('region', as_index=False)[years].mean()
    
    return result_df, years

# Обрабатываем оба файла
df_rural, years = process_child_mortality_fixed(df_child_mort_rural)
df_urban, _ = process_child_mortality_fixed(df_child_mort_urban)

# Объединяем сельскую и городскую смертность за 2020
df_child_mort = pd.merge(
    df_rural[['region', '2020']], 
    df_urban[['region', '2020']], 
    on='region', 
    how='outer',
    suffixes=('_rural', '_urban')
)
    
# Считаем суммарную младенческую смертность
df_child_mort['child_mortality_total_2020'] = (
    df_child_mort['2020_rural'].fillna(0) + df_child_mort['2020_urban'].fillna(0)
)
    
df_child_mort_2020 = df_child_mort[['region', 'child_mortality_total_2020']].copy()
    
print(f"\nПолучено регионов: {len(df_child_mort_2020)}")
print(df_child_mort_2020.head(3))

# Добавляем в df_agg
df_agg = df_agg.merge(df_child_mort_2020, on='region', how='left')
print(f"\nРазмер df_agg после добавления: {df_agg.shape}")


Получено регионов: 98
                                          region  child_mortality_total_2020
0  Агинский Бурятский округ (Забайкальский край)                        0.00
1                                 Алтайский край                       97.00
2                               Амурская область                       42.00

Размер df_agg после добавления: (82, 31)


In [23]:
# --- 2.4.1 Первичный обзор финального датасета (df_agg) ---

print("=" * 50)
print("ИТОГОВЫЙ ДАТАСЕТ: ПЕРВИЧНЫЙ ОБЗОР")
print("=" * 50)

print(f"\nРазмерность датасета: {df_agg.shape}")
print(f"Количество регионов: {df_agg['region'].nunique()}")
print(f"Количество признаков: {len(df_agg.columns) - 1}") # минус колонка 'region'

print("\nПервые 5 строк датасета:")
display(df_agg.head())

print("\nИнформация о колонках и типах данных:")
df_agg.info()

print("\nОсновные описательные статистики:")
display(df_agg.describe(include='all').T)  # .T для транспонирования для лучшего отображения

ИТОГОВЫЙ ДАТАСЕТ: ПЕРВИЧНЫЙ ОБЗОР

Размерность датасета: (82, 31)
Количество регионов: 82
Количество признаков: 30

Первые 5 строк датасета:


,region,poverty_percent,income_per_capita,grp_per_capita,retail_turnover_per_capita,year,workers_percent,disabled_total,births,morbidity_rate_2020,несовершеннолетние,ранее_совершавшие,ранее_судимые,группа_лиц,группа_лиц_предварительный_сговор,организованная_группа,преступное_сообщество,в_состоянии_опьянения_всего,алкогольное_опьянение,наркотическое_опьянение,housing_satisfied_pct,housing_area_total_per_person,housing_area_living_per_person,housing_rooms_per_household,welfare_expense_share,poor_total_pct,poor_children_pct,poor_working_pct,poor_pensioners_pct,population_2020,child_mortality_total_2020
0,Алтайский край,17.50,23864.00,291156.90,153605.00,2020,81.74,173935.00,19976.00,116618.39,887.00,15400.00,7278.00,64.00,901.00,87.00,2.00,9396.00,46.00,0.00,83.20,25.90,18.00,2.40,29.70,100.00,31.40,59.40,9.20,2317153.00,97.00
1,Амурская область,15.20,35499.00,571362.10,245233.00,2020,85.54,64692.00,7853.00,93183.50,431.00,6001.00,3078.00,52.00,409.00,57.00,0.00,3123.00,108.00,2.00,77.30,23.20,17.30,2.30,21.70,100.00,38.40,56.90,4.70,790044.00,42.00
2,Архангельская область,11.50,51926.00,2139378.57,241780.00,2020,100.00,44243.50,19000.00,123076.22,571.00,6818.00,3648.00,49.00,994.00,54.00,0.00,4222.00,32.00,4.00,82.15,28.05,18.90,2.55,17.90,100.00,29.35,63.70,6.95,757690.00,22.00
3,Астраханская область,15.60,25199.00,526950.90,174527.00,2020,93.44,43609.00,10910.00,62153.14,343.00,5395.00,2622.00,42.00,680.00,80.00,19.00,2534.00,70.00,2.00,77.50,24.90,19.20,2.30,31.40,100.00,42.10,52.40,5.50,1005782.00,64.00
4,Белгородская область,7.20,32841.00,646569.00,237076.00,2020,95.92,199929.00,12351.00,72498.65,219.00,4599.00,2787.00,15.00,505.00,26.00,129.00,2251.00,49.00,0.00,86.30,25.10,17.80,2.60,16.10,100.00,NaN,NaN,NaN,1549151.00,47.00



Информация о колонках и типах данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 82 entries, 0 to 81
Data columns (total 31 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   region                             82 non-null     object 
 1   poverty_percent                    82 non-null     float64
 2   income_per_capita                  81 non-null     float64
 3   grp_per_capita                     82 non-null     float64
 4   retail_turnover_per_capita         82 non-null     float64
 5   year                               82 non-null     int64  
 6   workers_percent                    82 non-null     float64
 7   disabled_total                     82 non-null     float64
 8   births                             82 non-null     float64
 9   morbidity_rate_2020                82 non-null     float64
 10  несовершеннолетние                 81 non-null     float64
 11  ранее_совершавшие    

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
region,82,82,Алтайский край,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
poverty_percent,82.00,NaN,NaN,NaN,14.05,5.00,5.90,10.85,13.40,16.05,31.70
income_per_capita,81.00,NaN,NaN,NaN,32057.00,12800.31,16877.00,24864.00,28152.00,32841.00,89541.00
grp_per_capita,82.00,NaN,NaN,NaN,582723.63,478145.13,142199.70,329947.50,455560.20,568065.22,2439663.45
retail_turnover_per_capita,82.00,NaN,NaN,NaN,197357.38,56388.87,50713.00,164775.50,196640.50,231179.75,408674.00
year,82.00,NaN,NaN,NaN,2020.00,0.00,2020.00,2020.00,2020.00,2020.00,2020.00
workers_percent,82.00,NaN,NaN,NaN,86.16,9.50,58.20,81.08,87.02,93.33,100.00
disabled_total,82.00,NaN,NaN,NaN,134699.70,142749.98,1638.00,57826.50,96430.00,179584.75,1009535.00
births,82.00,NaN,NaN,NaN,18190.85,20336.58,537.00,6837.25,10611.50,19897.75,123520.00
morbidity_rate_2020,82.00,NaN,NaN,NaN,82219.89,17237.38,48165.66,69948.90,80294.75,94903.62,123076.22


In [24]:
# --- 2.4.2 Анализ пропусков в данных ---

print("=" * 50)
print("АНАЛИЗ ПРОПУСКОВ")
print("=" * 50)

# Расчет количества и доли пропусков
missing_data = df_agg.isnull().sum().reset_index()
missing_data.columns = ['column', 'missing_count']
missing_data['missing_percent'] = (missing_data['missing_count'] / len(df_agg)) * 100

# Сортируем по убыванию доли пропусков и показываем только те, где есть пропуски
missing_data = missing_data[missing_data['missing_count'] > 0].sort_values('missing_percent', ascending=False)

print("\nКолонки с пропусками:")
display(missing_data)

АНАЛИЗ ПРОПУСКОВ

Колонки с пропусками:


,column,missing_count,missing_percent
27,poor_working_pct,11,13.41
26,poor_children_pct,11,13.41
28,poor_pensioners_pct,11,13.41
22,housing_area_living_per_person,6,7.32
21,housing_area_total_per_person,6,7.32
23,housing_rooms_per_household,6,7.32
20,housing_satisfied_pct,6,7.32
11,ранее_совершавшие,1,1.22
10,несовершеннолетние,1,1.22
2,income_per_capita,1,1.22


### 2.4.3 Обработка пропусков в данных

Мы выявили несколько групп пропусков в датасете. Для обеспечения полноты данных и возможности проведения дальнейшего анализа была разработана следующая стратегия их заполнения:

| Группа признаков | Количество пропусков | Стратегия обработки | Обоснование |
|------------------|----------------------|---------------------|-------------|
| **Структура бедности** (`poor_children_pct`, `poor_working_pct`, `poor_pensioners_pct`) | 11 регионов (13.4%) | Заполнение данными за 2019 год из файла `poverty_socdem_2019.xls` | Показатели структуры бедности относительно стабильны год от года, использование данных предыдущего года предпочтительнее статистического заполнения |
| **Жилищные условия** (`housing_satisfied_pct`, `housing_area_total_per_person`, `housing_area_living_per_person`, `housing_rooms_per_household`) | 6 регионов (7.3%) | Заполнение медианными значениями по всем регионам | Распределение показателей жилищных условий близко к нормальному, медиана является робастной оценкой центральной тенденции |
| **Единичные пропуски** (показатели преступности, доходов, расходов на соцполитику) | 1 регион (1.2%) | Заполнение медианными значениями | Незначительное количество пропусков позволяет использовать медиану без существенного смещения распределений |

In [25]:
# --- Заполняем пропуски в данных малоимущего населения по возрасту данными за 2019 год ---

df_socdem_2019 = pd.read_excel(
    DATA_PATH + 'poverty_socdem_2019.xls',
    sheet_name='Данные',
    header=None
)

# Пропускаем первые 2 строки
df_socdem_2019.columns = df_socdem_2019.iloc[2].fillna('').astype(str).tolist()
df_socdem_2019 = df_socdem_2019.iloc[3:].copy()

# Переименовываем первую колонку в 'region' и очищаем
df_socdem_2019 = df_socdem_2019.rename(columns={df_socdem_2019.columns[0]: 'region'})
df_socdem_2019 = df_socdem_2019[['region', 'Все население', 'Дети в возрасте до 16 лет', 
                       'Население трудоспособного возраста', 'Население старше трудоспособного возраста']].copy()

# Переименовываем колонки (временные имена, чтобы не конфликтовать с основными)
df_socdem_2019 = df_socdem_2019.rename(columns={
    'Все население': 'poor_total_pct_2019',
    'Дети в возрасте до 16 лет': 'poor_children_pct_2019',
    'Население трудоспособного возраста': 'poor_working_pct_2019',
    'Население старше трудоспособного возраста': 'poor_pensioners_pct_2019'
})

# Нормализация регионов
df_socdem_2019['region'] = df_socdem_2019['region'].apply(normalize_region)
df_socdem_2019 = df_socdem_2019.dropna(subset=['region']).reset_index(drop=True)

# Преобразование в числовой формат
for col in ['poor_total_pct_2019', 'poor_children_pct_2019', 'poor_working_pct_2019', 'poor_pensioners_pct_2019']:
    df_socdem_2019[col] = pd.to_numeric(df_socdem_2019[col], errors='coerce')

# Группировка на случай дубликатов
df_socdem_2019 = df_socdem_2019.groupby('region', as_index=False).mean()

print(f"Получено регионов за 2019: {len(df_socdem_2019)}")
print(df_socdem_2019.head())

# Добавляем временные колонки с данными за 2019 в df_agg
df_agg = df_agg.merge(df_socdem_2019, on='region', how='left')

# Заполняем пропуски в основных колонках данными за 2019 год
for col in ['poor_total_pct', 'poor_children_pct', 'poor_working_pct', 'poor_pensioners_pct']:
    col_2019 = col + '_2019'
    df_agg[col] = df_agg[col].fillna(df_agg[col_2019])

# Удаляем временные колонки с данными за 2019
df_agg = df_agg.drop(columns=['poor_total_pct_2019', 'poor_children_pct_2019', 
                              'poor_working_pct_2019', 'poor_pensioners_pct_2019'])

print(f"Пропуски заполнены данными за 2019 год")
print(f"Размер df_agg после заполнения: {df_agg.shape}")

Получено регионов за 2019: 82
                  region  poor_total_pct_2019  poor_children_pct_2019  poor_working_pct_2019  poor_pensioners_pct_2019
0         Алтайский край               100.00                   38.70                  57.40                      3.90
1       Амурская область               100.00                   33.90                  58.90                      7.20
2  Архангельская область               100.00                   36.63                  57.07                      6.33
3   Астраханская область               100.00                   43.40                  51.90                      4.70
4   Белгородская область               100.00                    0.00                   0.00                      0.00
Пропуски заполнены данными за 2019 год
Размер df_agg после заполнения: (82, 31)


In [26]:
# --- Жилищные условия ---
# - медиана
housing_cols = ['housing_satisfied_pct', 'housing_area_total_per_person', 
                'housing_area_living_per_person', 'housing_rooms_per_household']

for col in housing_cols:
    median_val = df_agg[col].median()
    df_agg[col] = df_agg[col].fillna(median_val)

# --- Единичные пропуски ---
#- медиана
single_cols = ['income_per_capita', 'welfare_expense_share'] + \
              ['несовершеннолетние', 'ранее_совершавшие', 'ранее_судимые', 
               'группа_лиц', 'группа_лиц_предварительный_сговор', 'организованная_группа',
               'преступное_сообщество', 'в_состоянии_опьянения_всего', 
               'алкогольное_опьянение', 'наркотическое_опьянение']

for col in single_cols:
    if col in df_agg.columns and df_agg[col].isnull().sum() > 0:
        median_val = df_agg[col].median()
        df_agg[col] = df_agg[col].fillna(median_val)

# Финальная проверка
print("\n" + "=" * 50)
print("ФИНАЛЬНАЯ ПРОВЕРКА")
print("=" * 50)
print(f"Всего пропусков после обработки: {df_agg.isnull().sum().sum()}")
print(f"Размерность обработанного датасета: {df_agg.shape}")


ФИНАЛЬНАЯ ПРОВЕРКА
Всего пропусков после обработки: 0
Размерность обработанного датасета: (82, 31)
